# Train PVT-MoE
Expects `ROOT/table.csv` and `ROOT/{id}.npy` (see README).

In [ ]:
import math, numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader
from dataset import PVTDataset, phase_stats
from model import PVTMoE
from losses import total_loss

ROOT, DEVICE = "data", "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS, FREEZE, BATCH, PRETRAINED = 100, 5, 32, True
torch.manual_seed(42); np.random.seed(42)

In [ ]:
df = pd.read_csv(f"{ROOT}/table.csv")
tr = df[df.split == "train"]
mean, std = phase_stats(tr, ROOT)
train_dl = DataLoader(PVTDataset(tr, ROOT, mean, std, train=True), BATCH, shuffle=True)

In [ ]:
model = PVTMoE(PRETRAINED).to(DEVICE)
backbone = {id(p) for p in model.enc.parameters()}

def groups():
    out = []
    for is_bb, lr in ((False, 1e-4), (True, 1e-5)):
        ps = [p for p in model.parameters() if (id(p) in backbone) == is_bb]
        out += [dict(params=[p for p in ps if p.ndim > 1], weight_decay=0.05, base=lr, bb=is_bb),
                dict(params=[p for p in ps if p.ndim <= 1], weight_decay=0.0, base=lr, bb=is_bb)]
    return out

opt = torch.optim.AdamW(groups())
scaler = torch.amp.GradScaler(enabled=DEVICE == "cuda")

def lr_scale(e, start, warm):
    k = e - start
    if k < 0: return 0.0
    if k < warm: return (k + 1) / warm
    return 0.5 * (1 + math.cos(math.pi * (k - warm) / max(1, EPOCHS - start - warm)))

In [ ]:
for e in range(EPOCHS):
    for g in opt.param_groups:
        g["lr"] = g["base"] * (lr_scale(e, FREEZE, 2) if g["bb"] else lr_scale(e, 0, 5))
    for p in model.enc.parameters():
        p.requires_grad = e >= FREEZE
    model.train(); run = []
    for x, y in train_dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
            loss = total_loss(*model(x), y)
        opt.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); run.append(loss.item())
    print(f"epoch {e:3d}  loss {np.mean(run):.3f}")
torch.save(dict(model=model.state_dict(), mean=mean, std=std), "model.pt")